# Starlink Obstruction Analysis

Reproduces **Figures 3, 9, 10, 12, 15** and **Tables 2–5** from the NINeS 2026 paper  
*"What Obstructed Skies Teach Us About Satellite Internet"*

Two Starlink dishes are co-located (control = `pi1`, test = `pi2`).  
The test dish is physically obstructed with a metal sheet in the south-east direction.  
This notebook analyses:
1. How often the two dishes connect to *different* satellites (responsive routing)
2. Whether extra pixel trajectories in the obstruction map signal a mid-slot handover
3. Hourly RTT and loss patterns for obstructed and unobstructed periods
4. Second-granularity RTT + loss during a responsive routing event
5. CDF of RTT difference (control − test) for same vs different satellite periods

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'process'))

# Processing scripts (IRTT, TLE, pipeline): see process/
from data_loading import *
from satellite_matching import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from tqdm import tqdm

est = timezone('US/Eastern')

## Daily Percentage of Instances Where Dishes Connect to Different Satellites

For each 15-second observation window we have a satellite match for both dishes.  
We flag windows where `pi1_sat != pi2_sat` (different satellite) and aggregate by day.

In [ ]:
# Load satellite match results for the full obstructed period
start_date = "2024-12-16"
end_date   = "2025-01-22"
sat_match_data = read_sat_match_data(start_date, end_date)

In [ ]:
# Flag each window as same-satellite or different-satellite
diff_sat = sat_match_data.copy()
diff_sat['diff_sat'] = (diff_sat['pi1_sat'] != diff_sat['pi2_sat']).astype(int)
diff_sat['same_sat'] = 1 - diff_sat['diff_sat']
diff_sat['date'] = pd.to_datetime(diff_sat['date'], utc=True).dt.tz_convert('US/Eastern').dt.normalize()

# Aggregate: count same/diff per day, compute daily percentage
daily = diff_sat.groupby('date')[['diff_sat', 'same_sat']].sum().reset_index()
daily['total'] = daily['diff_sat'] + daily['same_sat']
daily['percent_diff_sat'] = (daily['diff_sat'] / daily['total']) * 100

# Drop Dec 15 (incomplete data)
daily = daily[daily['date'] != pd.Timestamp('2024-12-15', tz='US/Eastern')]

print(daily[['date', 'diff_sat', 'total', 'percent_diff_sat']].to_string(index=False))

In [ ]:
# daily diff-sat percentage
import matplotlib.pyplot as plt

plt.figure(figsize=(5,3))
plt.plot(daily['date'], daily['percent_diff_sat'])
# plt.scatter(daily['date'], daily['percent_diff_sat'])
plt.xlabel('Date')
plt.ylim(0, 30)
plt.grid()
plt.ylabel('Different Satellite (%)')
plt.xticks(rotation=45)
plt.tight_layout()
# plt.savefig('figures/diff_sat_daily.pdf', bbox_inches='tight')
plt.show()

## OBS Map Pixel Analysis (Tables 2–5)

The Starlink app writes a 15-second trajectory to the obstruction map each handover window.  
When a dish switches satellite *mid-window* (responsive routing), a second partial trajectory  
appears in the map. We classify each window's residual pixels (after removing the longest  
trajectory blob) as one of:

- **no px** — no residual pixels; normal single-trajectory window  
- **split trajectory** — residual pixels that belong to neither the previous nor the next window's trajectory  
- **underflow** — residual pixels spatially close to the *next* window's trajectory (beginning leaked in early)  
- **overflow** — residual pixels spatially close to the *previous* window's trajectory (tail carried over)

We test three proximity thresholds (0 = exact adjacency, 1 = 1-px gap, 2 = 2-px gap).

In [ ]:
# Load data for the obstructed period used in Tables 2–5
start_date_str = "2025-01-04"
end_date_str   = "2025-01-14"
sat_match_data = read_sat_match_data(start_date_str, end_date_str)

start_date = datetime(2025, 1, 4,  0, 0, 0, 0, est)
end_date   = datetime(2025, 1, 14, 23, 59, 59, 0, est)
rst_obsmap_dict = get_rst_obsmap_dict(start_date, end_date)

### Helper Functions: Proximity and Multi-Trajectory Detection

In [ ]:
def in_proximity(pa, pb, threshold=1):
    """True if two pixels are within `threshold` pixels of each other in both axes."""
    return abs(pa[0] - pb[0]) <= threshold and abs(pa[1] - pb[1]) <= threshold


def close_lines(points_in, trajectory, threshold=1):
    """True if any pixel in `points_in` is within `threshold` of any pixel in `trajectory`."""
    for p in points_in:
        for t in trajectory:
            if in_proximity(p, t, threshold=threshold):
                return True
    return False


def classify_residual_pixels(curr_img, next_img, prev_img, prev_prev_img, threshold=1):
    """
    After removing the longest trajectory blob from `curr_img`, classify the
    remaining pixels as: no px, underflow, overflow, or split trajectory.

    Parameters
    ----------
    curr_img      : PIL Image — current 15-s window obstruction map
    next_img      : PIL Image — next window (for underflow check)
    prev_img      : PIL Image — previous window (for overflow check)
    prev_prev_img : PIL Image — two windows back (overflow of prev)
    threshold     : int — pixel proximity tolerance

    Returns
    -------
    (bool, str) — (has_extra_trajectory, category)
    """
    curr_rm = remove_longest_block(curr_img, find_longest_contiguous_non_black(curr_img))

    if len(find_longest_contiguous_non_black(curr_rm)) == 0:
        return True, 'no px'   # no residual pixels; normal window

    curr_rm_nb  = np.argwhere(np.array(curr_rm) > 0)
    prev_nb     = np.argwhere(np.array(prev_img) > 0)
    next_nb     = np.argwhere(np.array(next_img) > 0)

    # Overflow: residual is spatially close to the previous window's trajectory
    if close_lines(curr_rm_nb, prev_nb, threshold):
        return True, 'overflow'

    # Underflow: residual is spatially close to the next window's trajectory
    if close_lines(curr_rm_nb, next_nb, threshold):
        return True, 'underflow'

    # Residual belongs to neither neighbour window → genuine second trajectory
    return False, 'split trajectory'


def classify_window(date, rst_obsmap_dict, threshold=1, pi='pi2'):
    """
    Classify a single 15-s observation window for dish `pi` at `date`.
    Loads curr/prev/next images and delegates to classify_residual_pixels.
    Returns (bool, category_str).
    """
    try:
        curr_img      = get_image_small(date,                               rst_obsmap_dict, pi)
        prev_img      = get_image_small(date - pd.Timedelta(seconds=15),    rst_obsmap_dict, pi)
        next_img      = get_image_small(date + pd.Timedelta(seconds=15),    rst_obsmap_dict, pi)
        prev_prev_img = get_image_small(date - pd.Timedelta(seconds=30),    rst_obsmap_dict, pi)
        return classify_residual_pixels(curr_img, next_img, prev_img, prev_prev_img, threshold)
    except Exception:
        return False, 'file not found'

### Tables 2 & 3: Pixel Classification — All Connections

The first table uses *all* observation windows (both same-satellite and different-satellite).  
The second table uses only windows where the two dishes connected to *different* satellites.  
Both tables show control (`pi1`) and test (`pi2`) columns across thresholds 0, 1, 2.

In [ ]:
# --- All connections, both dishes, thresholds 0/1/2 ---
all_dates      = sat_match_data['date'].to_list()
diff_sat_dates = sat_match_data[sat_match_data['pi1_sat'] != sat_match_data['pi2_sat']]['date'].to_list()

print("=" * 60)
print("All connections")
print("=" * 60)
for threshold in range(0, 3):
    for pi in ['pi1', 'pi2']:
        counts = {}
        for date in tqdm(all_dates, desc=f"thres={threshold} {pi}"):
            _, category = classify_window(date, rst_obsmap_dict, threshold, pi)
            counts[category] = counts.get(category, 0) + 1
        total = len(all_dates)
        print(f"\nThreshold={threshold}  Terminal={pi}  Total={total}")
        for k, v in sorted(counts.items()):
            print(f"  {k:<20} {v:6d}  ({100*v/total:.2f}%)")

In [ ]:
# --- Different-satellite windows only, both dishes, thresholds 0/1/2 ---
print("=" * 60)
print("Different-satellite windows only")
print("=" * 60)
for threshold in range(0, 3):
    for pi in ['pi1', 'pi2']:
        counts = {}
        for date in tqdm(diff_sat_dates, desc=f"thres={threshold} {pi}"):
            _, category = classify_window(date, rst_obsmap_dict, threshold, pi)
            counts[category] = counts.get(category, 0) + 1
        total = len(diff_sat_dates)
        print(f"\nThreshold={threshold}  Terminal={pi}  Total={total}")
        for k, v in sorted(counts.items()):
            print(f"  {k:<20} {v:6d}  ({100*v/total:.2f}%)")

### Tables 4 & 5: First-Connection Classification During Different-Satellite Instances

When the two dishes connect to different satellites, we look at what trajectory the *test dish*  
had in the immediately prior and current window and ask: did it start on the *same* satellite  
as the control dish before switching?

- **Strict**: overflow/underflow pixels are excluded (treated as noise)  
- **Relaxed**: overflow/underflow pixels count as same-satellite if spatially close to the detected trajectory

In [ ]:
def classify_first_connection(date, rst_obsmap_dict, threshold=1, relaxed=True):
    """
    For a diff-sat window, classify what trajectory the test dish (pi2) was on
    just before and at the switch point.

    Returns (bool, category_str) where category is one of:
      'curr pi1 match'  — test dish's current extra trajectory matches control
      'prev pi1 match'  — test dish's previous extra trajectory matches control
      'curr no px'      — no residual pixels in current window
      'prev no px'      — no residual pixels in previous window
      'curr no match'   — residual exists but does not match control
      'prev no match'   — previous residual does not match control
      'curr overflow'   — residual is overflow from prior window (strict only)
      'curr underflow'  — residual is underflow into next window (strict only)
      'prev overflow'   — same for prev window (strict only)
      'prev prev ctrl'  — prev residual matches the control's *own* prior trajectory (strict only)
      'prev underflow'  — prev residual matches current window trajectory (strict only)
      'file not found'  — image missing
    """
    try:
        curr_img      = get_image_small(date,                               rst_obsmap_dict, 'pi2')
        ctrl_img      = get_image_small(date,                               rst_obsmap_dict, 'pi1')
        prev_ctrl_img = get_image_small(date - pd.Timedelta(seconds=15),    rst_obsmap_dict, 'pi1')
        next_img      = get_image_small(date + pd.Timedelta(seconds=15),    rst_obsmap_dict, 'pi2')
        prev_img      = get_image_small(date - pd.Timedelta(seconds=15),    rst_obsmap_dict, 'pi2')
        prev_prev_img = get_image_small(date - pd.Timedelta(seconds=30),    rst_obsmap_dict, 'pi2')
    except Exception:
        return False, 'file not found'

    def check_window(img, ctrl, next_w, prev_w, prev_prev_w, window_label):
        rm = remove_longest_block(img, find_longest_contiguous_non_black(img))
        if len(find_longest_contiguous_non_black(rm)) == 0:
            return False, f'{window_label} no px'

        rm_nb   = np.argwhere(np.array(rm) > 0)
        ctrl_nb = np.argwhere(np.array(ctrl) > 0)

        if not relaxed:
            # Strict: flag overflow/underflow as not a real second trajectory
            if close_lines(rm_nb, np.argwhere(np.array(next_w) > 0), threshold):
                return False, f'{window_label} underflow'
            if close_lines(rm_nb, np.argwhere(np.array(prev_w) > 0), threshold):
                return False, f'{window_label} overflow'
            if window_label == 'prev':
                prev_prev_nb  = np.argwhere(np.array(prev_prev_w) > 0)
                prev_ctrl_nb  = np.argwhere(np.array(prev_ctrl_img) > 0)
                if close_lines(rm_nb, prev_prev_nb, threshold):
                    return False, 'prev overflow'
                if close_lines(rm_nb, prev_ctrl_nb, threshold):
                    return False, 'prev prev ctrl'
                curr_lb = np.argwhere(np.array(find_longest_contiguous_non_black(curr_img)) > 0)
                if close_lines(rm_nb, curr_lb, threshold):
                    return False, 'prev underflow'

        if close_lines(rm_nb, ctrl_nb, threshold):
            return True, f'{window_label} pi1 match'
        return False, f'{window_label} no match'

    result = check_window(curr_img, ctrl_img, next_img, prev_img, prev_prev_img, 'curr')
    if result[0]:
        return result
    return check_window(prev_img, ctrl_img, curr_img, prev_prev_img, None, 'prev')

In [ ]:
# --- Tables 4 & 5: run strict and relaxed for diff-sat windows ---
for relaxed, table_name in [(False, "Strict"), (True, "Relaxed")]:
    print("=" * 60)
    print(table_name)
    print("=" * 60)
    for threshold in range(0, 3):
        counts = {}
        for date in tqdm(diff_sat_dates, desc=f"thres={threshold}"):
            _, category = classify_first_connection(date, rst_obsmap_dict, threshold, relaxed)
            counts[category] = counts.get(category, 0) + 1
        total = len(diff_sat_dates)
        print(f"\nThreshold={threshold}  Total={total}")
        for k, v in sorted(counts.items()):
            print(f"  {k:<22} {v:6d}  ({100*v/total:.2f}%)")

---
## RTT and Loss Analysis (Figures 3, 9, 12, 15)

The following cells reproduce the RTT/loss figures from the paper. Additional imports are added here.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import numpy as np

## Hourly RTT and Loss

The SE-obstructed period (Jan 17–27) and unobstructed baseline (Jan 29 – Feb 7) are each loaded below; run the matching load + plot cells for the desired period.

Run each pair of load + plot cells for the desired period by setting `start_date` / `end_date`.

### SE-Obstructed Period (Jan 17–27)

In [ ]:
start_date = "2025-01-17"
end_date   = "2025-01-27"

rtt_data_df = read_rtt_data(start_date, end_date)

hourly_rtt = rtt_data_df.rename(columns={'date': 'timestamp'}).copy()
hourly_rtt['timestamp'] = hourly_rtt['timestamp'].dt.floor('h')
hourly_rtt = hourly_rtt.groupby('timestamp').agg(
    rtt_pi1_mean   = ('rtt_pi1', 'mean'),
    rtt_pi1_median = ('rtt_pi1', 'median'),
    rtt_pi1_p05    = ('rtt_pi1', lambda x: x.quantile(0.05)),
    rtt_pi1_p95    = ('rtt_pi1', lambda x: x.quantile(0.95)),
    rtt_pi2_mean   = ('rtt_pi2', 'mean'),
    rtt_pi2_median = ('rtt_pi2', 'median'),
    rtt_pi2_p05    = ('rtt_pi2', lambda x: x.quantile(0.05)),
    rtt_pi2_p95    = ('rtt_pi2', lambda x: x.quantile(0.95)),
).reset_index()
print(hourly_rtt.head())

In [ ]:
# hourly RTT with p05–p95 shaded bands
pi2_color = 'tab:orange'
fig, ax = plt.subplots(figsize=(6, 3))

ax.fill_between(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p05'], hourly_rtt['rtt_pi1_p95'],
                alpha=0.3, color='blue', label='Control RTT\nP05-P95 Range')
ax.fill_between(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p05'], hourly_rtt['rtt_pi2_p95'],
                alpha=0.3, color=pi2_color, label='Test RTT\nP05-P95 Range')

ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_median'], color='darkblue', label='Control RTT\nMedian')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_median'], color=pi2_color,  label='Test RTT\nMedian')

ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p05'], color='blue',    linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p95'], color='blue',    linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p05'], color=pi2_color, linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p95'], color=pi2_color, linewidth=1, alpha=0.7, linestyle='--')

ax.set_ylim(30, 80)
ax.set_xlabel('Time')
ax.set_ylabel('RTT (ms)')
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.xaxis.set_major_locator(mdates.HourLocator(interval=24))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=45)
plt.tight_layout()
# plt.savefig('figures/rtt_obstructed.pdf', bbox_inches='tight')
plt.show()

In [ ]:
start_date = "2025-01-17"
end_date   = "2025-01-26"

loss_data = get_loss_data(start_date, end_date)
loss_df = loss_data.rename(columns={'date': 'timestamp'}).copy()
loss_df['timestamp'] = loss_df['timestamp'].dt.floor('h')
loss_df = loss_df.groupby('timestamp').agg(
    pi1_loss_sum=('pi1_loss', 'sum'),
    pi2_loss_sum=('pi2_loss', 'sum'),
).reset_index()

In [ ]:
total_loss = 5 * 100 * 60 * 60

fig, ax = plt.subplots(figsize=(5, 3))
plt.rc('font', size=11)
ax.plot(loss_df['timestamp'], loss_df['pi1_loss_sum'] / (total_loss / 100), label='Control Loss')
ax.plot(loss_df['timestamp'], -loss_df['pi2_loss_sum'] / (total_loss / 100), label='Test Loss')
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{abs(x):.0f}"))
plt.xticks(rotation=45)
ax.set_xlabel('Time')
ax.set_ylabel('Loss (%)')
ax.legend(loc='upper right')
ax.grid()
plt.tight_layout()
# plt.savefig('figures/loss_obstructed.pdf', bbox_inches='tight')
plt.show()

### Unobstructed Baseline (Jan 29 – Feb 7)

In [ ]:
start_date = "2025-01-29"
end_date   = "2025-02-07"

rtt_data_df = read_rtt_data(start_date, end_date)

hourly_rtt = rtt_data_df.rename(columns={'date': 'timestamp'}).copy()
hourly_rtt['timestamp'] = hourly_rtt['timestamp'].dt.floor('h')
hourly_rtt = hourly_rtt.groupby('timestamp').agg(
    rtt_pi1_mean   = ('rtt_pi1', 'mean'),
    rtt_pi1_median = ('rtt_pi1', 'median'),
    rtt_pi1_p05    = ('rtt_pi1', lambda x: x.quantile(0.05)),
    rtt_pi1_p95    = ('rtt_pi1', lambda x: x.quantile(0.95)),
    rtt_pi2_mean   = ('rtt_pi2', 'mean'),
    rtt_pi2_median = ('rtt_pi2', 'median'),
    rtt_pi2_p05    = ('rtt_pi2', lambda x: x.quantile(0.05)),
    rtt_pi2_p95    = ('rtt_pi2', lambda x: x.quantile(0.95)),
).reset_index()
# Drop last 12 hours (incomplete data at period boundary)
hourly_rtt = hourly_rtt[hourly_rtt['timestamp'] < hourly_rtt['timestamp'].max() - pd.Timedelta(hours=12)]
print(hourly_rtt.head())

In [ ]:
# hourly RTT with p05–p95 shaded bands
pi2_color = 'tab:orange'
fig, ax = plt.subplots(figsize=(6, 3))

ax.fill_between(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p05'], hourly_rtt['rtt_pi1_p95'],
                alpha=0.3, color='blue', label='Control RTT\nP05-P95 Range')
ax.fill_between(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p05'], hourly_rtt['rtt_pi2_p95'],
                alpha=0.3, color=pi2_color, label='Test RTT\nP05-P95 Range')

ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_median'], color='darkblue', label='Control RTT\nMedian')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_median'], color=pi2_color,  label='Test RTT\nMedian')

ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p05'], color='blue',    linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p95'], color='blue',    linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p05'], color=pi2_color, linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p95'], color=pi2_color, linewidth=1, alpha=0.7, linestyle='--')

ax.set_ylim(30, 80)
ax.set_xlabel('Time')
ax.set_ylabel('RTT (ms)')
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.xaxis.set_major_locator(mdates.HourLocator(interval=24))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=45)
plt.tight_layout()
# plt.savefig('figures/rtt_unobstructed.pdf', bbox_inches='tight')
plt.show()

In [ ]:
start_date = "2025-01-29"
end_date   = "2025-02-07"

loss_data = get_loss_data(start_date, end_date)
loss_df = loss_data.rename(columns={'date': 'timestamp'}).copy()
loss_df['timestamp'] = loss_df['timestamp'].dt.floor('h')
loss_df = loss_df.groupby('timestamp').agg(
    pi1_loss_sum=('pi1_loss', 'sum'),
    pi2_loss_sum=('pi2_loss', 'sum'),
).reset_index()

In [ ]:
total_loss = 5 * 100 * 60 * 60

fig, ax = plt.subplots(figsize=(5, 3))
plt.rc('font', size=11)
ax.plot(loss_df['timestamp'], loss_df['pi1_loss_sum'] / (total_loss / 100), label='Control Loss')
ax.plot(loss_df['timestamp'], -loss_df['pi2_loss_sum'] / (total_loss / 100), label='Test Loss')
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{abs(x):.0f}"))
plt.xticks(rotation=45)
ax.set_xlabel('Time')
ax.set_ylabel('Loss (%)')
ax.legend(loc='upper right')
ax.grid()
plt.tight_layout()
# plt.savefig('figures/loss_unobstructed.pdf', bbox_inches='tight')
plt.show()

## Second-Granularity RTT + Loss Spike

Shows a specific responsive routing event on Jan 1, 2025 where both dishes briefly connect to different satellites. The event is identified by finding contiguous windows where pi1_sat ≠ pi2_sat, then selecting the 22nd such chunk (index 21).

In [ ]:
est = timezone('US/Eastern')
event_start = datetime(2025, 1, 1, 0, 0, 0, 0, est)
event_end   = datetime(2025, 1, 1, 23, 59, 59, 0, est)

rst_obsmap_dict = get_rst_obsmap_dict(event_start, event_end)
rtt_data_df     = read_rtt_data("2025-01-01", "2025-01-01")
loss_data       = get_loss_data("2025-01-01", "2025-01-01")
sat_match_data  = read_sat_match_data("2025-01-01", "2025-01-03")
all_sat_data    = sat_match_data.sort_values("date").reset_index(drop=True)

In [ ]:
# Find contiguous diff-sat event chunks
diff_indices = all_sat_data.index[all_sat_data["pi1_sat"] != all_sat_data["pi2_sat"]]
extended = set(diff_indices)
for idx in diff_indices:
    if idx > 0: extended.add(idx - 1)
    if idx < len(all_sat_data) - 1: extended.add(idx + 1)

diff_sat_data = all_sat_data.loc[sorted(extended)].reset_index(drop=True)

chunks, current_chunk = [], []
for _, row in diff_sat_data.iterrows():
    if not current_chunk:
        current_chunk.append(row['date'])
    else:
        current_chunk.append(row['date'])
        if row['pi1_sat'] == row['pi2_sat']:
            chunks.append(current_chunk)
            current_chunk = []

# Keep only fully contiguous chunks (no gaps)
filtered_chunks = []
for chunk in chunks:
    contiguous = all(
        pd.to_datetime(chunk[i]) + pd.Timedelta(seconds=15) == pd.to_datetime(chunk[i+1])
        for i in range(len(chunk)-1)
    )
    if contiguous:
        filtered_chunks.append(chunk)

print(f"Found {len(filtered_chunks)} contiguous diff-sat event chunks")

In [ ]:
# plot event chunk 21 (index 21) at second granularity
time_chunks = filtered_chunks[21]
event_date  = time_chunks[1]  # middle window of the chunk

start_time = event_date - pd.Timedelta(seconds=16)
end_time   = event_date

rtt_chunk  = rtt_data_df[(rtt_data_df['date'] > start_time) & (rtt_data_df['date'] < end_time)]
loss_chunk = loss_data[(loss_data['date'] > start_time) & (loss_data['date'] < end_time)]

# Align loss to RTT timestamps
aligned_loss = pd.merge(rtt_chunk[['date']], loss_chunk[['date', 'pi1_loss', 'pi2_loss']],
                        on='date', how='left').fillna(0)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(rtt_chunk['date'], rtt_chunk['rtt_pi1'], color='tab:blue',   label='Control RTT')
ax.plot(rtt_chunk['date'], rtt_chunk['rtt_pi2'], color='tab:orange', label='Test RTT')
ax.axvline(pd.Timestamp("2025-01-01 07:01:12-05:00"), color='black',               label='Handover 1')
ax.axvline(pd.Timestamp("2025-01-01 07:01:16-05:00") + pd.Timedelta(milliseconds=200),
           color='black', linestyle='--', label='Handover 2')

ax2 = ax.twinx()
ax2.scatter(aligned_loss['date'], aligned_loss['pi1_loss'], color='tab:cyan', label='Control Loss', s=3)
ax2.scatter(aligned_loss['date'], aligned_loss['pi2_loss'], color='tab:red',  label='Test Loss',    s=3)
ax2.set_ylim(0, 6)
ax2.set_ylabel('Loss')

ax.xaxis.set_major_locator(mdates.SecondLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%S'))
ax.set_xlabel('Timestamp (seconds)')
ax.set_ylabel('RTT (ms)')
ax.grid()

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
plt.tight_layout()
# plt.savefig('figures/rtt_loss_spike.pdf', bbox_inches='tight')
plt.show()

show_obs_image(time_chunks[0], rst_obsmap_dict)

## CDF of RTT Difference — Same vs Different Satellite

Compares the distribution of (Control − Test) RTT for 15-second windows where both dishes are on the **same** satellite vs **different** satellites. Uses the SE-obstructed period (Jan 17–27, 2025).

RTT measurements are bucketed into 15-second intervals matching the sat-match timestamps, then the median and P05 differences are plotted.

In [ ]:
start_date = "2025-01-17"
end_date   = "2025-01-27"

sat_match_data = read_sat_match_data(start_date, end_date)
rtt_data_df    = read_rtt_data(start_date, end_date)

same_sat = sat_match_data[sat_match_data['pi1_sat'] == sat_match_data['pi2_sat']]['date'].tolist()
diff_sat = sat_match_data[sat_match_data['pi1_sat'] != sat_match_data['pi2_sat']]['date'].tolist()
print(f"Same satellite periods: {len(same_sat)}")
print(f"Different satellite periods: {len(diff_sat)}")

In [ ]:
# Bucket each RTT measurement into its 15-second sat-match window
rtt = rtt_data_df.rename(columns={'date': 'timestamp'}).copy()
rtt['timestamp'] = pd.to_datetime(rtt['timestamp'])

def assign_bucket(ts):
    s = ts.second
    base = ts.floor('min')
    if s < 11:   return (ts - pd.Timedelta(seconds=s+1)).floor('min') + pd.Timedelta(seconds=56)
    elif s < 26: return base + pd.Timedelta(seconds=11)
    elif s < 41: return base + pd.Timedelta(seconds=26)
    elif s < 56: return base + pd.Timedelta(seconds=41)
    else:        return base + pd.Timedelta(seconds=56)

rtt['timestamp'] = rtt['timestamp'].apply(assign_bucket)

irtt_pdf = rtt.groupby('timestamp').agg(
    rtt_pi1_median=('rtt_pi1', 'median'),
    rtt_pi1_p05   =('rtt_pi1', lambda x: x.quantile(0.05)),
    rtt_pi2_median=('rtt_pi2', 'median'),
    rtt_pi2_p05   =('rtt_pi2', lambda x: x.quantile(0.05)),
).reset_index()

irtt_pdf['diff_median'] = irtt_pdf['rtt_pi1_median'] - irtt_pdf['rtt_pi2_median']
irtt_pdf['diff_p05']    = irtt_pdf['rtt_pi1_p05']    - irtt_pdf['rtt_pi2_p05']

irtt_same = irtt_pdf[irtt_pdf['timestamp'].isin(same_sat)]
irtt_diff = irtt_pdf[irtt_pdf['timestamp'].isin(diff_sat)]
print(f"Same-sat RTT rows: {len(irtt_same)}, Diff-sat RTT rows: {len(irtt_diff)}")

In [ ]:
# CDF of RTT difference (control − test)
fig, ax = plt.subplots(figsize=(7, 3))

for data, color, sat_label in [(irtt_same, 'tab:blue', 'Same Satellites'),
                                (irtt_diff, 'tab:orange', 'Different Satellites')]:
    for col, style, stat_label in [('diff_median', '-', 'Median'), ('diff_p05', '--', 'P05')]:
        vals = data[col].dropna()
        if len(vals) == 0:
            continue
        sorted_vals = np.sort(vals)
        ax.plot(sorted_vals, np.linspace(0, 1, len(sorted_vals)),
                color=color, linestyle=style, linewidth=2,
                label=f'{sat_label}\n({stat_label})')

ax.set_xlabel('RTT Difference (Control - Test) [ms]')
ax.set_ylabel('Cumulative Probability')
ax.set_xlim(-20, 20)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), frameon=True)
plt.tight_layout()
# plt.savefig('figures/rtt_diff_cdf.pdf', bbox_inches='tight')
plt.show()

---
## Satellite Trajectory Analysis (Figures 6, 7, 11)

DTW-based trajectory comparison and azimuth/elevation distributions.

## DTW Normalization Comparison

Five variants for normalizing polar coordinates (elevation, azimuth) before computing DTW distance between co-located dish trajectories:
- `default` — raw Cartesian conversion from (elevation, azimuth)
- `max75` — elevation rescaled to [0, 1] over the [25°, 100°] visible range
- `max90` — elevation divided by 90
- `cos` — radial weight = cos(elevation), reflecting sky solid angle
- `sin` — radial weight = sin(elevation)

`cos` was selected as the best-performing method (lowest DTW distance for same-satellite pairs).

In [ ]:
# DTW normalization variants — functions imported from matching_utils via data_loading
DTW_VARIANTS = {
    'default':       calculate_dtw_error,
    'max75':         max75_normalize,
    'max90':         max90_normalize,
    'cos_normalize': cos_normalize,
    'sin_normalize': sin_normalize,
}

In [ ]:
est = timezone('US/Eastern')
abs_start_date = datetime(2025, 1, 29, 0, 0, 0, 0, est)
abs_end_date   = datetime(2025, 2, 7, 23, 59, 59, 0, est)
day_num = (abs_end_date - abs_start_date).days

pi_error_10days = {key: {} for key in DTW_VARIANTS}

for i in range(day_num + 1):
    start_date = abs_start_date + timedelta(days=i)
    end_date   = start_date + timedelta(days=1) - timedelta(seconds=1)
    print(start_date.date())

    rst_obsmap_dict = get_rst_obsmap_dict(start_date, end_date)
    pi_coord_dict   = get_pi_coord_dict(rst_obsmap_dict)

    for d in tqdm(pi_coord_dict['pi1']):
        pi1_cd = pi_coord_dict['pi1'][d]
        pi2_cd = pi_coord_dict['pi2'][d]
        for key, fn in DTW_VARIANTS.items():
            pi_error_10days[key][d] = fn(pi1_cd, pi2_cd)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3))
plt.rc('font', size=11)
for key in pi_error_10days.keys():
    error_list = sorted([e for e in pi_error_10days[key].values() if e < 1000])
    if key == 'cos_normalize':
        key = 'cos'
    if key == 'sin_normalize':
        key = 'sin'
    plt.plot(error_list, np.linspace(0, 1, len(error_list)), label=key)
plt.ylim(0.9, 1)
plt.xlim(0, 100)
plt.legend()
plt.xlabel('DTW distance difference')
plt.ylabel('CDF')
plt.grid()
# plt.savefig('figures/dtw_normalization_cdf.pdf', bbox_inches='tight')
plt.show()

## Azimuth vs Elevation — Unobstructed Period

Scatter plot of the test dish's (pi2) satellite connection points during the unobstructed period (Jan 29–30, 2025). Shows where in the sky the dish connects when the SE obstruction has no effect.

In [ ]:
start_date = "2025-01-29"
end_date   = "2025-01-30"
sat_match_data = read_sat_match_data(start_date, end_date)

fig, ax = plt.subplots(figsize=(5, 3))
plt.rc('font', size=11)
ax.scatter(sat_match_data['pi2_first_az'], sat_match_data['pi2_first_ele'], color='red', s=1)
ax.yaxis.set_major_locator(ticker.MultipleLocator(50))
ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
ax.set_xlim(0, 90)
ax.set_xlabel('Elevation')
ax.set_ylabel('Azimuth')
ax.grid()
plt.tight_layout()
# plt.savefig('figures/az_el_unobstructed.pdf', bbox_inches='tight')
plt.show()

## CDF of Azimuth — Obstructed vs Unobstructed

Compares the azimuth distribution of the first satellite connection for both dishes:
- **Unobstructed** (Jan 28 – Feb 6, 2025): both dishes free to connect in any direction
- **SE-obstructed** (Jan 8–18, 2025): pi2 has a steel sheet blocking the SE direction

Dashed vertical lines mark the obstruction boundaries at azimuth 93.4° and 146.6°. The test dish (pi2) shifts its connections away from the blocked region when obstructed.

In [ ]:
# Unobstructed period: Jan 28 – Feb 6
sat_match_data_no = read_sat_match_data("2025-01-28", "2025-02-06")
# SE-obstructed period: Jan 8 – 18
sat_match_data_se = read_sat_match_data("2025-01-08", "2025-01-18")

fig, ax = plt.subplots(figsize=(5, 3))
plt.rc('font', size=11)

pi1_no = sat_match_data_no['pi1_first_ele'].sort_values()
pi2_no = sat_match_data_no['pi2_first_ele'].sort_values()
ax.plot(pi1_no, np.linspace(0, 1, len(pi1_no), endpoint=False), label='Control unobstructed', color='blue')
ax.plot(pi2_no, np.linspace(0, 1, len(pi2_no), endpoint=False), label='Test unobstructed', color='red')

pi1_se = sat_match_data_se['pi1_first_ele'].sort_values()
pi2_se = sat_match_data_se['pi2_first_ele'].sort_values()
ax.plot(pi1_se, np.linspace(0, 1, len(pi1_se), endpoint=False), label='Control with\n test obstructed', color='blue', linestyle='dotted')
ax.plot(pi2_se, np.linspace(0, 1, len(pi2_se), endpoint=False), label='Test obstructed', color='red', linestyle='dotted')

AOE = [93.4, 146.6]
ax.axvline(x=AOE[0], color='black', linestyle='dashed', alpha=0.5)
ax.axvline(x=AOE[1], color='black', linestyle='dashed', alpha=0.5)

ax.set_xlabel('Azimuth (degrees)')
ax.set_ylabel('CDF')
ax.legend()
ax.grid()
plt.tight_layout()
# plt.savefig('figures/azimuth_cdf.pdf', bbox_inches='tight')
plt.show()